In [169]:
import re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

DATA_DIR = '/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/'
HOLDOUT_START = 1827
LAGS_AR = [1,5]
LAGS_FUND = [1,5]
VAL_FRAC = .15
SEEDS = (0,1,2)

train        = pd.read_csv(f'{DATA_DIR}/train.csv')
train_labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
target_pairs = pd.read_csv(f'{DATA_DIR}/target_pairs.csv')

train = train.sort_values('date_id').reset_index(drop=True)
train_labels = train_labels.sort_values('date_id').reset_index(drop=True)
assert (train['date_id'].values == train_labels['date_id'].values).all(), "Mismatched date_id between train and labels"

target_cols = [c for c in train_labels.columns if c != 'date_id']
n_days = len(train)

print(f'train {train.shape}  labels {train_labels.shape} pairs {target_pairs.shape}')
print(f"total days: {n_days}, {len(target_cols)} targets")

train (1961, 558)  labels (1961, 425) pairs (424, 3)
total days: 1961, 424 targets


In [170]:
def parse_pair(pair_str):
    s = pair_str.strip()
    tokens = re.split(r'\s*([+-])\s*', s)
    result, sign = [], 1
    if tokens[0]:
        result.append((sign, tokens[0]))
    i = 1
    while i < len(tokens):
        op, ticker = tokens[i], tokens[i + 1]
        result.append((1 if op == "+" else -1, ticker))
        i+= 2
    return result

def get_exchange(ticker):
    if ticker.startswith('US_Stock_'):
        return 'US_STOCK'
    return ticker.split("_")[0]

rows = []
for _, r in target_pairs.iterrows():
    components = parse_pair(r['pair'])
    tickers = [t for _, t in components]
    exchanges = set(get_exchange(t) for t in tickers)
    rows.append({
        'target': r['target'],
        'lag': r['lag'],
        'pair_raw': r['pair'],
        'tickers': tickers,
        'exchange_key': "_".join(sorted(exchanges)),
    })
target_info = pd.DataFrame(rows)
print(target_info.head())
print(f'\nUnique exchange keys: {sorted(target_info["exchange_key"].unique())}')
print(f'Lag distribution: {target_info["lag"].value_counts().to_dict()}')


     target  lag                                        pair_raw                                          tickers  exchange_key
0  target_0    1                           US_Stock_VT_adj_close                          [US_Stock_VT_adj_close]      US_STOCK
1  target_1    1            LME_PB_Close - US_Stock_VT_adj_close            [LME_PB_Close, US_Stock_VT_adj_close]  LME_US_STOCK
2  target_2    1                     LME_CA_Close - LME_ZS_Close                     [LME_CA_Close, LME_ZS_Close]           LME
3  target_3    1                     LME_AH_Close - LME_ZS_Close                     [LME_AH_Close, LME_ZS_Close]           LME
4  target_4    1  LME_AH_Close - JPX_Gold_Standard_Futures_Close  [LME_AH_Close, JPX_Gold_Standard_Futures_Close]       JPX_LME

Unique exchange keys: ['FX', 'FX_JPX', 'FX_LME', 'JPX_LME', 'JPX_US_STOCK', 'LME', 'LME_US_STOCK', 'US_STOCK']
Lag distribution: {1: 106, 2: 106, 3: 106, 4: 106}


In [171]:
def classify_ticker(t):
    fams = set()
    if 'JPX_Gold' in t:                                 fams.add('gold')
    if 'JPX_Platinum' in t:                             fams.add('platinum')
    if 'Rubber' in t:                                   fams.add('rubber')
    if 'LME_CA' in t:                                   fams.add('copper')
    if 'LME_AH' in t:                                   fams.add('aluminum')
    if 'LME_PB' in t:                                   fams.add('lead')
    if 'LME_ZS' in t:                                   fams.add('zinc')
    # US-listed proxies that effectively belong to the same family
    if any(s in t for s in ['_GLD_','_IAU_','_GDX_','_NEM_','_AEM_','_GOLD_','_KGC_','_FNV_','_WPM_','_SLV_']):
        fams.add('gold')
    if any(s in t for s in ['_FCX_','_SCCO_']):
        fams.add('copper')
    return fams

def families_for_tickers(tickers):
    fams = set()
    for t in tickers:
        fams |= classify_ticker(t)
    return fams

print('family coverage of targets:')
for ekey in sorted(target_info['exchange_key'].unique())[:8]:
    sample = target_info[target_info['exchange_key'] == ekey].iloc[0]
    print(f'  {ekey:<25} -> {sample["tickers"]} -> {families_for_tickers(sample["tickers"])} ')

family coverage of targets:
  FX                        -> ['FX_ZARUSD'] -> set() 
  FX_JPX                    -> ['JPX_Platinum_Standard_Futures_Close', 'FX_CADCHF'] -> {'platinum'} 
  FX_LME                    -> ['FX_AUDJPY', 'LME_PB_Close'] -> {'lead'} 
  JPX_LME                   -> ['LME_AH_Close', 'JPX_Gold_Standard_Futures_Close'] -> {'aluminum', 'gold'} 
  JPX_US_STOCK              -> ['US_Stock_IEMG_adj_close', 'JPX_Gold_Standard_Futures_Close'] -> {'gold'} 
  LME                       -> ['LME_CA_Close', 'LME_ZS_Close'] -> {'copper', 'zinc'} 
  LME_US_STOCK              -> ['LME_PB_Close', 'US_Stock_VT_adj_close'] -> {'lead'} 
  US_STOCK                  -> ['US_Stock_VT_adj_close'] -> set() 


In [172]:
def lret(s):
    return np.log(s / s.shift(1))

def build_engineered(train, lags=LAGS_FUND):
    #returns (features_df, family_to_cols)
    out = {}
    fam_cols = {f: [] for f in ['gold', 'platinum', 'rubber', 'copper', 'aluminum', 'lead', 'zinc']}

    # Base series (computed once)
    series = {
        'usdjpy_ret':     lret(train['FX_USDJPY']),
        'gld_ret':        lret(train['US_Stock_GLD_adj_close']),
        'gsr':            train['US_Stock_GLD_adj_close'] / train['US_Stock_SLV_adj_close'].replace(0, np.nan),
        'plat_gold':      train['JPX_Platinum_Standard_Futures_Close'] / train['JPX_Gold_Standard_Futures_Close'].replace(0, np.nan),
        'xle_ret':        lret(train['US_Stock_XLE_adj_close']),
        'fxi_ret':        lret(train['US_Stock_FXI_adj_close']),
        'copper_gold':    train['LME_CA_Close']     / train['JPX_Gold_Standard_Futures_Close'].replace(0, np.nan),
        'fcx_ret':        lret(train['US_Stock_FCX_adj_close']),
        'xlb_ret':        lret(train['US_Stock_XLB_adj_close']),
    }
    series['gsr_ret']         = lret(series['gsr'])
    series['plat_gold_ret']   = lret(series['plat_gold'])
    series['copper_gold_ret'] = lret(series['copper_gold'])

    # Map each base series to the families that should see it
    membership = {
        'usdjpy_ret':       ['gold','platinum','rubber'],            # JPY translation
        'gld_ret':          ['gold'],
        'gsr':              ['gold'],
        'gsr_ret':          ['gold'],
        'plat_gold':        ['platinum'],
        'plat_gold_ret':    ['platinum'],
        'xle_ret':          ['rubber'],
        'fxi_ret':          ['rubber','copper'],
        'copper_gold':      ['copper','aluminum','lead','zinc'],     # Dr. Copper as macro pulse
        'copper_gold_ret':  ['copper','aluminum','lead','zinc'],
        'fcx_ret':          ['copper'],
        'xlb_ret':          ['aluminum','lead','zinc'],
    }

    # Apply lags and assign to families
    for base, fams in membership.items():
        for k in lags:
            name = f'{base}_lag{k}'
            out[name] = series[base].shift(k)
            for f in fams:
                fam_cols[f].append(name)

    feats = pd.DataFrame(out).ffill().fillna(0)
    return feats, fam_cols

eng_df, fam_cols = build_engineered(train)
print(f'eng_df shape: {eng_df.shape}')
print('columns per family:')
for f, cs in fam_cols.items():
    print(f' {f:<10} {len(cs):>3} cols e.g. {cs[:3]}')

eng_df shape: (1961, 24)
columns per family:
 gold         8 cols e.g. ['usdjpy_ret_lag1', 'usdjpy_ret_lag5', 'gld_ret_lag1']
 platinum     6 cols e.g. ['usdjpy_ret_lag1', 'usdjpy_ret_lag5', 'plat_gold_lag1']
 rubber       6 cols e.g. ['usdjpy_ret_lag1', 'usdjpy_ret_lag5', 'xle_ret_lag1']
 copper       8 cols e.g. ['fxi_ret_lag1', 'fxi_ret_lag5', 'copper_gold_lag1']
 aluminum     6 cols e.g. ['copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1']
 lead         6 cols e.g. ['copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1']
 zinc         6 cols e.g. ['copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1']


In [173]:
def own_target_features(target_name, train_labels, target_lag, ar_lags=LAGS_AR, vol_windows=(5,20)):
    """
    Lagged returns and realized vol of the target itself.
    target_lag: the per-target forecast horizon. Shifts must be >= target_lag
    to avoid leakage (you don't observe y[t] until t+target_lag).
    """
    y = train_labels[target_name]
    parts = []
    for k in ar_lags:
        shift = max(k, target_lag)
        parts.append(y.shift(shift).rename(f'own_lag{k}'))
    y_known = y.shift(target_lag)
    for w in vol_windows:
        parts.append(y_known.rolling(w).std().rename(f'own_vol{w}'))
    if len(vol_windows) >= 2:
        s = y_known.rolling(vol_windows[0]).std()
        l = y_known.rolling(vol_windows[-1]).std()
        parts.append((s / (l + 1e-8)).rename('own_vol_ratio'))
    return pd.concat(parts, axis=1).fillna(0)

def _resolve_price_col(ticker, cols):
    #map a ticker string from target_pairs to matching column in train
    cset = set(cols)
    for cand in (ticker, f'{ticker}_Close', f'{ticker}_adj_close', f'{ticker}_close'):
        if cand in cset:
            return cand
    for c in cols:
        if c.startswith(ticker):
            return c
    return None

def ticker_features(target_row, train, lags=LAGS_FUND):
    #lagged log returns for each constituent ticker in target pair
    target_lag = target_row['lag']
    parts = []
    missing = []
    for tk in target_row['tickers']:
        col = _resolve_price_col(tk, train.columns)
        if col is None:
            missing.append(tk)
            continue
        s = lret(train[col])
        for k in lags:
            shift = max(k, target_lag)
            parts.append(s.shift(shift).rename(f'{tk}_ret_lag{shift}'))
    if not parts:
        return pd.DataFrame(index=train.index), missing
    return pd.concat(parts, axis=1).ffill().fillna(0), missing

def features_for_target(target_row, eng_df, fam_cols, train_labels, train,
                         include_all_macro=False, include_macro=True):
    # constituent ticker features (raw price log-returns from train.csv)
    X_tk, missing = ticker_features(target_row, train)

    # macro / family-specific cross-asset features
    parts = [X_tk.reset_index(drop=True)]
    if include_macro or include_all_macro:
        if include_all_macro:
            X_cross = eng_df.copy()
        else:
            fams = families_for_tickers(target_row['tickers'])
            cols = sorted({c for f in fams for c in fam_cols.get(f, [])})
            X_cross = eng_df[cols] if cols else pd.DataFrame(index=eng_df.index)
        parts.append(X_cross.reset_index(drop=True))

    # own target AR features (lagged target log-returns from train_labels)
    X_own = own_target_features(target_row['target'], train_labels, target_row['lag'])
    parts.append(X_own.reset_index(drop=True))

    return pd.concat(parts, axis=1), missing

In [174]:
inspect_targets = []
for ekey in ['LME', 'JPX', 'US_STOCK_LME', 'US_STOCK_JPX']:
    cand = target_info[target_info['exchange_key'] == ekey]
    if len(cand) > 0:
        inspect_targets.append(cand.iloc[0])

for tr in inspect_targets:
    X, missing = features_for_target(tr, eng_df, fam_cols, train_labels, train)
    print(f'\n=== {tr["target"]}  pair="{tr["pair_raw"]}"  lag={tr["lag"]} ===')
    print(f'  tickers:   {tr["tickers"]}')
    print(f'  families:  {families_for_tickers(tr["tickers"])}')
    print(f'  X shape:   {X.shape}')
    print(f'  columns:   {list(X.columns)}')


=== target_2  pair="LME_CA_Close - LME_ZS_Close"  lag=1 ===
  tickers:   ['LME_CA_Close', 'LME_ZS_Close']
  families:  {'copper', 'zinc'}
  X shape:   (1961, 19)
  columns:   ['LME_CA_Close_ret_lag1', 'LME_CA_Close_ret_lag5', 'LME_ZS_Close_ret_lag1', 'LME_ZS_Close_ret_lag5', 'copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1', 'copper_gold_ret_lag5', 'fcx_ret_lag1', 'fcx_ret_lag5', 'fxi_ret_lag1', 'fxi_ret_lag5', 'xlb_ret_lag1', 'xlb_ret_lag5', 'own_lag1', 'own_lag5', 'own_vol5', 'own_vol20', 'own_vol_ratio']


In [175]:
target_X = {}
target_y = {}
target_cols_per_target = {}
all_missing = {}

INCLUDE_MACRO = True  # set True to restore family-specific cross-asset features

for _, tr in target_info.iterrows():
    X, missing = features_for_target(tr, eng_df, fam_cols, train_labels, train,
                                     include_macro=INCLUDE_MACRO)
    target_X[tr['target']] = X.values.astype(np.float32)
    target_y[tr['target']] = train_labels[tr['target']].values.astype(np.float32)
    target_cols_per_target[tr['target']] = X.columns.tolist()
    if missing:
        all_missing[tr['target']] = missing

n_feat_dist = pd.Series([X.shape[1] for X in target_X.values()]).value_counts().sort_index()
print(f'built features for {len(target_X)} targets  (include_macro={INCLUDE_MACRO})')
print(f'feature count distribution:\n{n_feat_dist}')
if all_missing:
    print(f'\n{len(all_missing)} targets had unresolved tickers. Examples:')
    for t, m in list(all_missing.items())[:5]:
        print(f'  {t}: {m}')

built features for 424 targets  (include_macro=True)
feature count distribution:
7       4
15    253
17    128
19      9
21      6
23     18
25      6
Name: count, dtype: int64


In [176]:
def mitsui_metric(y_true, y_pred, verbose=False):
    daily_corrs = []
    for i in range(len(y_true)):
        mask = ~np.isnan(y_true[i])
        if mask.sum() < 2:
            continue
        corr, _ = spearmanr(y_true[i, mask], y_pred[i, mask])
        if not np.isnan(corr):
            daily_corrs.append(corr)
    arr = np.array(daily_corrs)
    std = arr.std()
    if std == 0:
        raise ZeroDivisionError("Standard deviation of daily correlations is zero, cannot compute metric.")
    sharpe = arr.mean() / std
    if verbose:
        print(f"mean {arr.mean():.4f}, std {std:.4f}, sharpe {sharpe:.4f} ndays {len(arr)}")
    return sharpe

In [177]:
train_mask = train['date_id'] < HOLDOUT_START
holdout_mask = train['date_id'] >= HOLDOUT_START

n_train = int(train_mask.sum())
split_idx = int(n_train * (1 - VAL_FRAC))
n_holdout = int(holdout_mask.sum())

y_hold_true = train_labels.loc[holdout_mask, target_cols].values
print (f"n_train {n_train}, val split_idx {split_idx}, holdout_days={n_holdout}, holdout {y_hold_true.shape}")

n_train 1827, val split_idx 1552, holdout_days=134, holdout (134, 424)


In [ ]:
#now do RNN instead of lightgpm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN     = 20
HIDDEN_SIZE = 64
N_LAYERS    = 2
LR          = 1e-3
EPOCHS      = 15
BATCH_SIZE  = 64
PATIENCE = 10
DROPOUT      = 0.2
WEIGHT_DECAY = 1e-4

print(f"Using device: {DEVICE}")
print(f'Seq len {SEQ_LEN}, hidden {HIDDEN_SIZE}, layers {N_LAYERS}, lr {LR}, epochs {EPOCHS}')

class SeqDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
        self.seq_len = seq_len

        self.valid = []
        for i in range(len(X)-seq_len):
            if not np.isnan(y[i + seq_len]):
                self.valid.append(i)
    def __len__(self):
        return len(self.valid)
    def __getitem__(self, idx):
        i = self.valid[idx]
        return self.X[i:i+self.seq_len], self.y[i + self.seq_len]

class LSTMRegressor(nn.Module):
    def __init__(self, n_feat, hidden=HIDDEN_SIZE, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0.0,)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(self.dropout(last)).squeeze(-1)

Using device: cpu
Seq len 20, hidden 64, layers 2, lr 0.001, epochs 5


In [179]:
def standardize(X_train, *others):
    mu = X_train.mean(axis=0, keepdims=True)
    sig = X_train.std(axis=0, keepdims=True) + 1e-8
    return [(arr - mu) / sig for arr in (X_train, *others)]

def train_one_target(target_name, verbose= True, seed=0):
    torch.manual_seed(seed)
    tr_row = target_info[target_info['target'] == target_name].iloc[0]
    X_full = target_X[target_name]
    y_full = target_y[target_name]
    n_feat = X_full.shape[1]
    cols = target_cols_per_target[target_name]
    
    if verbose:
        print(f'\n=== Training RNN for {target_name} ===')
        print(f'  pair: "{tr_row["pair_raw"]}"  lag={tr_row["lag"]}')
        print(f'  tickers: {tr_row["tickers"]}  families: {families_for_tickers(tr_row["tickers"])}')
        print(f'  X shape: {X_full.shape}, y shape: {y_full.shape}, n_feat={n_feat}')
        print(f'  feature columns ({len(cols)}): {cols}')
    
    X_tr = X_full[:split_idx]
    X_val = X_full[split_idx:n_train]
    X_ho = X_full[n_train:]
    y_tr = y_full[:split_idx]
    y_val = y_full[split_idx:n_train]
    y_ho = y_full[n_train:]

    X_tr, X_val, X_ho = standardize(X_tr, X_val, X_ho)

    if verbose:
        print(f'  splits  train={X_tr.shape}  val={X_val.shape}  holdout={X_ho.shape}')

    train_ds = SeqDataset(X_tr, y_tr, SEQ_LEN)
    val_ds = SeqDataset(X_val, y_val, SEQ_LEN)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    if verbose:
        print(f'  windows train={len(train_ds)}, val={len(val_ds)}')
    
    model = LSTMRegressor(n_feat).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    best_val = float('inf')
    best_state = None
    for epoch in range(EPOCHS):
        model.train()
        tr_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(yb)
        tr_loss /= max(len(train_ds), 1)

        model.eval()
        with torch.no_grad():
            v_loss = 0.0
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                v_loss += loss_fn(model(xb), yb).item() * len(yb)
            v_loss /= max(len(val_ds), 1)
        
        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        if verbose and (epoch ==0 or (epoch+1) % 5 == 0 or epoch == EPOCHS-1):
            print(f'  epoch {epoch+1:>2}/{EPOCHS}  train_mse={tr_loss:.5f}  val_mse={v_loss:.5f}')
    
    model.load_state_dict(best_state)

    X_hist = np.concatenate([X_tr, X_val, X_ho], axis=0)
    preds = np.zeros(len(X_ho), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for j in range(len(X_ho)):
            end = n_train + j
            window = X_hist[end-SEQ_LEN:end]
            xb = torch.from_numpy(window).unsqueeze(0).to(DEVICE)
            preds[j] = model(xb).item()
    if verbose:
        print(f'  holdout preds shape: {preds.shape}')
    return preds

#demo one target so it prints clearly
demo_pred = train_one_target('target_0', verbose=True)



=== Training RNN for target_0 ===
  pair: "US_Stock_VT_adj_close"  lag=1
  tickers: ['US_Stock_VT_adj_close']  families: set()
  X shape: (1961, 7), y shape: (1961,), n_feat=7
  feature columns (7): ['US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'own_lag1', 'own_lag5', 'own_vol5', 'own_vol20', 'own_vol_ratio']
  splits  train=(1552, 7)  val=(275, 7)  holdout=(134, 7)
  windows train=1432, val=236
  epoch  1/5  train_mse=0.00128  val_mse=0.00016
  epoch  5/5  train_mse=0.00025  val_mse=0.00006
  holdout preds shape: (134,)


In [180]:
#loop every target
predictions = np.full_like(y_hold_true, np.nan, dtype=np.float32)
for ti, t in enumerate(target_cols):
    p = train_one_target(t, verbose=(ti < 5))
    predictions[:, ti] = p
    if (ti + 1) % 25 == 0:
        print(f'  done {ti+1}/{len(target_cols)}')
print(f'predictions shape: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}')


=== Training RNN for target_0 ===
  pair: "US_Stock_VT_adj_close"  lag=1
  tickers: ['US_Stock_VT_adj_close']  families: set()
  X shape: (1961, 7), y shape: (1961,), n_feat=7
  feature columns (7): ['US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'own_lag1', 'own_lag5', 'own_vol5', 'own_vol20', 'own_vol_ratio']
  splits  train=(1552, 7)  val=(275, 7)  holdout=(134, 7)
  windows train=1432, val=236
  epoch  1/5  train_mse=0.00128  val_mse=0.00016
  epoch  5/5  train_mse=0.00025  val_mse=0.00006
  holdout preds shape: (134,)

=== Training RNN for target_1 ===
  pair: "LME_PB_Close - US_Stock_VT_adj_close"  lag=1
  tickers: ['LME_PB_Close', 'US_Stock_VT_adj_close']  families: {'lead'}
  X shape: (1961, 15), y shape: (1961,), n_feat=15
  feature columns (15): ['LME_PB_Close_ret_lag1', 'LME_PB_Close_ret_lag5', 'US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1', 'copper_gold_ret_lag5', 'xlb_

In [181]:
#score
print("HOLDOUT SCORE")
score = mitsui_metric(y_hold_true, predictions, verbose= True)
print(f"Sharpe: {score:.4f}")

HOLDOUT SCORE
mean 0.0064, std 0.1377, sharpe 0.0468 ndays 134
Sharpe: 0.0468


In [182]:
# ── Per-exchange-group breakdown ──────────────────────────────────────────────

print('=== Per-group Sharpe ===')
for ekey in sorted(target_info['exchange_key'].unique()):
    group_targets = target_info[target_info['exchange_key'] == ekey]['target'].tolist()
    y_idx   = [target_cols.index(t) for t in group_targets]
    g_true  = y_hold_true[:, y_idx]
    g_pred  = predictions[:, y_idx]
    try:
        s = mitsui_metric(g_true, g_pred)
        print(f'  {ekey:<20} {s:+.4f}  ({len(group_targets)} targets)')
    except ZeroDivisionError as e:
        print(f'  {ekey:<20} ERROR: {e}')

=== Per-group Sharpe ===
  FX                   -0.0598  (2 targets)
  FX_JPX               +0.3104  (46 targets)
  FX_LME               -0.0738  (86 targets)
  JPX_LME              -0.0075  (12 targets)
  JPX_US_STOCK         +0.0349  (87 targets)
  LME                  +0.0150  (8 targets)
  LME_US_STOCK         -0.1158  (181 targets)
  US_STOCK             +0.1898  (2 targets)
